# 04. 평가 & Grad-CAM 시각화

두 모델 비교, 혼동행렬, Grad-CAM, 실패 케이스 분석

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

from src import CLASSES, IDX_TO_CLASS
from src.dataset import PostureImageDataset, PosturePoseDataset
from src.models import build_resnet18, PoseMLP
from src.gradcam import GradCAM, overlay_cam, _TRANSFORM
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# CNN 테스트셋 평가
import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'src.evaluate', 'cnn',
     '--backbone', 'resnet18',
     '--model_path', 'models/cnn_resnet18_best.pt'],
    capture_output=True, text=True
)
print(result.stdout)

result2 = subprocess.run(
    [sys.executable, '-m', 'src.evaluate', 'pose',
     '--model_path', 'models/pose_mlp_best.pt'],
    capture_output=True, text=True
)
print(result2.stdout)

In [ ]:
# 두 모델 비교 표
reports = {}
for tag, label in [('cnn_resnet18', 'CNN (ResNet18)'), ('pose_mlp', 'Pose+MLP')]:
    path = f'results/metrics/{tag}_report.json'
    if os.path.exists(path):
        with open(path) as f:
            reports[label] = json.load(f)

rows = []
for model_name, data in reports.items():
    row = {'Model': model_name, 'Accuracy': f"{data['accuracy']:.4f}"}
    for cls in CLASSES:
        row[f'{cls} F1'] = f"{data['report'][cls]['f1-score']:.4f}"
    rows.append(row)

if rows:
    df_compare = pd.DataFrame(rows).set_index('Model')
    print(df_compare.to_string())
    df_compare.to_csv('results/metrics/model_comparison.csv')

In [ ]:
# 혼동행렬 나란히 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, tag, title in zip(axes, ['cnn_resnet18', 'pose_mlp'], ['CNN (ResNet18)', 'Pose+MLP']):
    img_path = f'results/figures/cm_{tag}.png'
    if os.path.exists(img_path):
        cm_img = plt.imread(img_path)
        ax.imshow(cm_img)
        ax.set_title(title)
    ax.axis('off')
plt.suptitle('혼동행렬 비교', fontsize=14)
plt.tight_layout()
plt.savefig('results/figures/cm_comparison.png', dpi=150)
plt.show()

In [ ]:
# Grad-CAM 시각화 (각 클래스 대표 이미지)
import random
from pathlib import Path

cnn_model = build_resnet18(pretrained=False)
if os.path.exists('models/cnn_resnet18_best.pt'):
    cnn_model.load_state_dict(torch.load('models/cnn_resnet18_best.pt', map_location=device))
    cnn_model = cnn_model.to(device)

    gradcam = GradCAM(cnn_model, cnn_model.layer4[-1])
    os.makedirs('results/figures/gradcam', exist_ok=True)

    fig, axes = plt.subplots(4, 3, figsize=(12, 16))
    for row, cls in enumerate(CLASSES):
        cls_dir = Path('data/raw') / cls
        if not cls_dir.exists():
            continue
        imgs = list(cls_dir.glob('*.jpg'))[:1] + list(cls_dir.glob('*.png'))[:1]
        if not imgs:
            continue
        img_path = random.choice(imgs[:3])
        pil_img = Image.open(img_path).convert('RGB')
        tensor = _TRANSFORM(pil_img).unsqueeze(0).to(device)
        cam, pred_idx = gradcam.generate(tensor)
        overlay = overlay_cam(pil_img, cam)

        axes[row][0].imshow(np.array(pil_img.resize((224,224))))
        axes[row][0].set_title(f'{cls}\nOriginal')
        axes[row][1].imshow(cam, cmap='jet')
        axes[row][1].set_title('Grad-CAM')
        axes[row][2].imshow(overlay)
        axes[row][2].set_title(f'Overlay (pred: {CLASSES[pred_idx]})')
        for ax in axes[row]:
            ax.axis('off')

    gradcam.remove_hooks()
    plt.suptitle('Grad-CAM 시각화 (ResNet18)', fontsize=14)
    plt.tight_layout()
    plt.savefig('results/figures/gradcam_summary.png', dpi=150)
    plt.show()
else:
    print('CNN 모델이 없습니다. 먼저 학습을 실행하세요.')

In [ ]:
# 추론 속도 비교 (CPU)
import time
from src.pose_extract import extract_keypoints, normalize_keypoints
import cv2

# 더미 이미지
dummy_img = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

# CNN 속도
cnn_cpu = build_resnet18(pretrained=False).cpu().eval()
if os.path.exists('models/cnn_resnet18_best.pt'):
    cnn_cpu.load_state_dict(torch.load('models/cnn_resnet18_best.pt', map_location='cpu'))
    from torchvision import transforms
    transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                                    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    pil_dummy = Image.fromarray(dummy_img[..., ::-1])
    t = time.time()
    N = 20
    for _ in range(N):
        with torch.no_grad():
            cnn_cpu(transform(pil_dummy).unsqueeze(0))
    cnn_ms = (time.time() - t) / N * 1000
    print(f'CNN (ResNet18) 추론 시간: {cnn_ms:.1f} ms/frame ({1000/cnn_ms:.1f} FPS)')

print('\n속도 비교는 webcam_demo.py 실행 시 실시간 측정 가능')